In [5]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

PARCELS_DIR = DATA_DIR / "parcels" / "processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"

parcels_assessor_path = PARCELS_DIR / "maricopa_parcels_with_assessor.gpkg"
village_clean_path = BOUNDARIES_DIR / "village_clean.gpkg"


parcels_assessor_path, village_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/maricopa_parcels_with_assessor.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/village_clean.gpkg'))

In [6]:
parcels = gpd.read_file(parcels_assessor_path)
village = gpd.read_file(village_clean_path)

print("Parcels CRS:", parcels.crs)
print("Village CRS:", village.crs)

Parcels CRS: EPSG:2868
Village CRS: EPSG:2868


In [7]:
# these are both already using the project CRS natively, but just in case:

PROJECT_CRS = "EPSG:2868"

if village.crs != PROJECT_CRS:
    village = village.to_crs(PROJECT_CRS)
    print("Reprojected Village to project CRS:", PROJECT_CRS)
if parcels.crs != PROJECT_CRS:
    parcels = parcels.to_crs(PROJECT_CRS)
    print("Reprojected parcels to project CRS:", PROJECT_CRS)

print("Village CRS:", village.crs)
print("Parcels CRS:", parcels.crs)

Village CRS: EPSG:2868
Parcels CRS: EPSG:2868


In [8]:
village.head()

,OBJECTID,NAME,geometry
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598..."
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551...."
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860..."
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89..."
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89..."


In [9]:
# Join parcels to Villages (inner or left join depending on what you want)

parcels_in_village = gpd.sjoin(
    parcels,
    village[["NAME", "geometry"]],
    how="left",
    predicate="intersects"
)

parcels_in_village.head()

,APN,Floor,LandSize,StartDate,Shp_Area,Shp_Length,PARCEL_NO,PROPERTYUSECODE,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,NEIGHBORHOODCODE,MARKETAREACODE,geometry,index_right,NAME
0,10101001C,1,7565.0,2008-09-22,7505.147490,350.125679,10101001C,5400.0,68700.0,68700.0,0.0,68700.0,11336.0,7565.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581182.478 886654.38 0, 5811...",NaN,NaN
1,10101010,1,195864.0,2008-09-22,195989.095940,1838.766520,10101010,9780.0,8667760.0,1196500.0,7471260.0,7435591.0,1115339.0,195864.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((582322.095 889960.815 0, 582...",NaN,NaN
2,10101011,1,95491.0,2008-09-22,95501.799733,1248.957937,10101011,9720.0,5190674.0,636700.0,4553974.0,3377292.0,506594.0,95491.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581500.432 889688.576 0, 581...",NaN,NaN
3,10101012,1,66960.0,2008-09-22,66976.092596,1014.388578,10101012,9705.0,553400.0,553400.0,0.0,305062.0,45759.0,66960.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581428.706 889618.792 0, 581...",NaN,NaN
4,10101014,1,126838.0,2008-09-22,126838.044773,1424.568514,10101014,9720.0,7074148.0,969800.0,6104348.0,2971142.0,445671.0,126838.0,NaN,1.0,6.0,"MULTIPOLYGON Z (((581359.726 889391.699 0, 581...",NaN,NaN


In [10]:
# verifying join results

# parcels_in_village['NAME'].isna().value_counts() # (False means matched to a Village)

# parcels_in_village[parcels_in_village['NAME'].notna()].head() # show head() for matched parcels only

# parcels_in_village.groupby('NAME').size() # count of parcels per Village

In [11]:
n_parcels = len(parcels)
n_joined  = len(parcels_in_village)
extra_rows = n_joined - n_parcels

print("Parcels:", n_parcels)
print("Joined:", n_joined)
print("Extra rows from multi-matches:", extra_rows)

Parcels: 1736192
Joined: 1736737
Extra rows from multi-matches: 545


In [12]:
match_counts = parcels_in_village.groupby(parcels_in_village.index).size()

# Only parcels that matched more than one village
multi_matches = match_counts[match_counts > 1]

print("Parcels with multiple village matches:", len(multi_matches))
multi_matches.head()

Parcels with multiple village matches: 539


70028    2
70035    2
70045    2
70046    2
70047    2
dtype: int64

In [13]:
dup_indices = multi_matches.index

dups = (
    parcels_in_village
    .loc[dup_indices, ["APN", "NAME"]]
    .sort_index()
)

dups.head(50)

,APN,NAME
70028,10461001,Laveen
70028,10461001,Estrella
70035,10463002,Laveen
70035,10463002,Estrella
70045,10464004,Laveen
70045,10464004,Estrella
70046,10464005,Laveen
70046,10464005,Estrella
70047,10464006,Laveen
70047,10464006,Estrella


In [ ]:
# Since there are 539 duplicate matches, the next step is to compute the intersection area for just those duplicates and assign it to whatever has the maximum overlap. I did enlist Copilot for this.

import geopandas as gpd
import pandas as pd

# Start from the joined layer
piv = parcels_in_village.copy()  # parcels_in_village from sjoin

# go back to initial processed layer
villages = gpd.read_file(village_clean_path)

# Identify parcels that matched multiple villages
dup_mask = piv.index.duplicated(keep=False)

dups = piv[dup_mask].copy()      # rows where parcel index appears >1 time
non_dups = piv[~dup_mask].copy() # rows with 0 or 1 village match

print("Total rows in join:      ", len(piv))
print("Rows with duplicates:    ", len(dups))
print("Rows without duplicates: ", len(non_dups))

# Attach village geometry to those duplicate rows
# Assumes `villages` index matches the `index_right` values used in sjoin
vill_geom = villages[["geometry"]].rename(columns={"geometry": "village_geom"})

dups = dups.join(vill_geom, on="index_right")

# Compute intersection area between each parcel and each candidate village
dups["intersect_area"] = dups.geometry.intersection(dups["village_geom"]).area

# For each parcel index, keep the row with the largest intersection area. Group by the index (parcel index); each group = one parcel, multiple villages
dups_best = (
    dups.sort_values("intersect_area", ascending=False)
        .groupby(level=0, as_index=False)
        .head(1)  # keep the first (largest area) per parcel index
)

# Rebuild a clean one-row-per-parcel join. Align columns with the original sjoin result (drop helper columns)
dups_best_aligned = dups_best[piv.columns]  # drop village_geom + intersect_area

piv_resolved = (
    pd.concat([non_dups, dups_best_aligned])
      .sort_index()
)

print("Final resolved rows:     ", len(piv_resolved))


Total rows in join:       1736737
Rows with duplicates:     1084
Rows without duplicates:  1735653
Final resolved rows:      1736192


In [15]:
# verifying join results

# piv_resolved['NAME'].isna().value_counts() # (False means matched to a Village)

# piv_resolved[piv_resolved['NAME'].notna()].head() # show head() for matched parcels only

# piv_resolved.groupby('NAME').size() # count of parcels per Village

In [16]:
piv_resolved_path = PARCELS_DIR / "parcels_assessor_village_no_duplicates.gpkg"
piv_resolved.to_file(piv_resolved_path, driver="GPKG")

print("Saved parcels + assessor + Village with no duplicates:", piv_resolved_path)

Saved parcels + assessor + Village with no duplicates: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\processed\parcels_assessor_village_no_duplicates.gpkg


In [17]:
parcels_in_village_path = PARCELS_DIR / "parcels_assessor_village.gpkg"
parcels_in_village.to_file(parcels_in_village_path, driver="GPKG")

print("Saved parcels + assessor + Village with duplicates:", parcels_in_village_path)

Saved parcels + assessor + Village with duplicates: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\processed\parcels_assessor_village.gpkg


In [20]:
piv_resolved[piv_resolved["NAME"].str.contains("Ahwat", case=False, na=False)]

,APN,Floor,LandSize,StartDate,Shp_Area,Shp_Length,PARCEL_NO,PROPERTYUSECODE,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,NEIGHBORHOODCODE,MARKETAREACODE,geometry,index_right,NAME
747884,30004011C,1,17852.0,2008-05-29,17598.468599,643.641287,30004011C,5400.0,80200.0,80200.0,0.0,36629.0,6044.0,17851.0,NaN,14.0,6.0,"MULTIPOLYGON Z (((630656.797 837228.32 0, 6304...",0.0,Ahwatukee Foothills
747885,30004013D,1,56773.0,2008-05-29,59773.808630,1167.378113,30004013D,5400.0,220300.0,220300.0,0.0,96998.0,16005.0,56773.0,NaN,14.0,6.0,"MULTIPOLYGON Z (((631538.096 835900.33 0, 6315...",0.0,Ahwatukee Foothills
747886,30004013F,1,288996.0,2008-05-29,290689.107240,3478.587265,30004013F,5400.0,911900.0,911900.0,0.0,380459.0,62776.0,288995.0,NaN,14.0,6.0,"MULTIPOLYGON Z (((631472.346 835976.638 0, 631...",0.0,Ahwatukee Foothills
747887,30004014A,1,549727.0,2008-05-29,545058.958278,6034.122499,30004014A,5400.0,1598300.0,1598300.0,0.0,652818.0,107715.0,549727.0,NaN,14.0,6.0,"MULTIPOLYGON Z (((633451.785 833680.443 0, 633...",0.0,Ahwatukee Foothills
747888,30004015D,1,435600.0,2008-05-29,430331.495049,2624.044263,30004015D,9700.0,274500.0,274500.0,0.0,274500.0,41175.0,435600.0,NaN,1.0,13.0,"MULTIPOLYGON Z (((631472.334 838535.138 0, 631...",0.0,Ahwatukee Foothills
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1704613,30175980A,1,133294.0,2023-02-02,133827.176366,2204.150628,30175980A,261.0,500.0,500.0,0.0,500.0,50.0,133294.0,NaN,13.0,1.0,"MULTIPOLYGON Z (((659046.283 842770.525 0, 659...",0.0,Ahwatukee Foothills
1704642,30159973,1,29400.0,2023-02-03,29364.516269,737.697606,30159973,9720.0,933100.0,933100.0,0.0,401233.0,60185.0,29400.0,NaN,13.0,1.0,"MULTIPOLYGON Z (((682814.839 844074.566 0, 682...",0.0,Ahwatukee Foothills
1705887,30036998,1,343321.0,2023-02-23,343320.997085,8330.672566,30036998,261.0,500.0,500.0,0.0,500.0,50.0,343321.0,NaN,13.0,1.0,"MULTIPOLYGON Z (((645026.731 833296.251 0, 644...",0.0,Ahwatukee Foothills
1731905,30170899A,1,3827.0,2024-06-26,3826.632448,278.039121,30170899A,9500.0,1831.0,1831.0,0.0,102.0,10.0,3827.0,NaN,4.0,13.0,"MULTIPOLYGON Z (((667734.302 835846.073 0, 667...",0.0,Ahwatukee Foothills
